In [1]:
!nvidia-smi | head -5
!pip install -q uv
!uv venv --python 3.12 /content/sgl && source /content/sgl/bin/activate && uv pip install --prerelease=allow sglang

Sat Oct 24 13:25:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: sgl
Activate with: source sgl/bin/activate
Using Python 3.13.16 environment at: /usr
Resolved 206 packages in 1.90s
error: Failed to build `outlines-core==0.1.26`
  cause: The build backend returned an error
  cause: Call to `setuptools.build_meta.build_wheel` failed (exit status: 1)

         [stdout]
         running bdist_wheel
         running build
         running build_py
         creating build/lib.linux-x86_64-cpython-313/outlines_core
         copying python/outlines_core/__init__.py -> build/li

In [2]:
!source /content/sgl/bin/activate && python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --port 30000 --dtype float16 --attention-backend triton --sampling-backend pytorch &

/content/sgl/bin/python: Error while finding module specification for 'sglang.launch_server' (ModuleNotFoundError: No module named 'sglang')


In [3]:
!pip install uv
!uv venv --python 3.12 /content/sgl
!uv pip install --python /content/sgl/bin/python --prerelease=allow sglang

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: sgl
? A virtual environment already exists at `sgl`. Do you want to replace it? [y/n] › yes

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to skip this prompt^C
Using Python 3.12.3 environment at: sgl
Resolved 207 packages in 286ms
Prepared 162 packages in 2m 43s
Installed 207 packages in 1.49s
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + airportsdata==20260905
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==2.3.0
 + anyio==4.15.1
 + apache-tvm-ffi==0.1.11
 + asttokens==3.0.2
 + attrs==26.1.0
 + blobfile==3.0.0
 + build==1.6.1
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + compressed-tensors==0.18.0
 + cryptography==50.0.2
 + cuda-bindings==13.4.4
 + cuda-core==1.2.2
 + cuda-pathfinder==1.8.4
 + cuda-python==13.4.4
 + cuda-tile==1.6.0rc5
 + cuda-toolkit==13.0.3.0
 + cython==3.3.0
 + dataset

In [4]:
!/content/sgl/bin/python -c "import sglang; print(sglang.__version__)"

0.5.22


In [5]:
!nohup /content/sgl/bin/python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --port 30000 --dtype float16 --attention-backend triton --sampling-backend pytorch > /content/sgl.log 2>&1 &

In [11]:
import subprocess, os
os.environ["PATH"] = "/content/sgl/bin:/usr/local/cuda/bin:" + os.environ["PATH"]
log = open("/content/sgl.log", "w")
p = subprocess.Popen(
    ["/content/sgl/bin/python", "-m", "sglang.launch_server",
     "--model-path", "Qwen/Qwen2.5-0.5B-Instruct", "--port", "30000",
     "--dtype", "float16", "--attention-backend", "triton",
     "--sampling-backend", "pytorch"],
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
print("started, pid", p.pid)

started, pid 10779


In [13]:
!tail -3 /content/sgl.log | cut -c1-300

[2026-10-24 14:00:01] INFO:     127.0.0.1:45730 - "POST /generate HTTP/1.1" 200 OK
[2026-10-24 14:00:01] INFO:     127.0.0.1:45738 - "POST /freeze_gc HTTP/1.1" 200 OK
[2026-10-24 14:00:01] The server is fired up and ready to roll!


In [14]:
!ps aux | grep launch_server | grep -v grep | wc -l

1


In [15]:
!curl -s http://127.0.0.1:30000/v1/chat/completions -H "Content-Type: application/json" -d '{"model":"Qwen/Qwen2.5-0.5B-Instruct","messages":[{"role":"user","content":"Ticket: My card was charged twice, very angry. Reply as JSON with category and sentiment."}],"temperature":0,"max_tokens":100,"response_format":{"type":"json_schema","json_schema":{"name":"t","schema":{"type":"object","properties":{"category":{"type":"string","enum":["billing","technical","account","shipping","other"]},"sentiment":{"type":"string","enum":["negative","neutral","positive"]}},"required":["category","sentiment"],"additionalProperties":false}}}}'

{"id":"2e9a814446ad42e9b42b51fbe576f8fa","object":"chat.completion","created":1792850528,"model":"Qwen/Qwen2.5-0.5B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"{\n  \"category\": \"technical\",\n  \"sentiment\": \"negative\"\n}","reasoning_content":null},"logprobs":null,"finish_reason":"stop","matched_stop":151645}],"usage":{"prompt_tokens":48,"total_tokens":66,"completion_tokens":18,"prompt_tokens_details":null,"reasoning_tokens":0},"metadata":{"weight_version":"default","weight_versions":[{"version":"default","start":0,"end":18}]}}